In [3]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier

# Read in the CSVs using pandas
df_train = pd.read_csv('./train.csv')
df_test = pd.read_csv('./test.csv')

def make_cleaned_data(df):
    # Create the departure year, month, day
    dates = df['FL_DATE'].astype(str).str.split('-')
    df['DEPARTURE_YEAR'] = dates.str[0].astype(int)
    df['DEPARTURE_MONTH'] = dates.str[1].astype(int)
    df['DEPARTURE_DAY'] = dates.str[2].astype(int)

    # Remove duplicate information
    df.drop(columns=['FL_DATE', 'MONTH', 'DAY_OF_MONTH'], axis=1, inplace=True)

    # Create the scheduled departure time columns
    df['Scheduled_DEP_EST'] = pd.to_datetime(df['Scheduled_DEP_EST'], errors='coerce')
    df['SCHEDULED_DEPARTURE_HOUR'] = df['Scheduled_DEP_EST'].dt.hour
    df['SCHEDULED_DEPARTURE_MINUTE'] = df['Scheduled_DEP_EST'].dt.minute

    # Create the scheduled arrival time columns
    df['Scheduled_ARR_EST'] = pd.to_datetime(df['Scheduled_ARR_EST'], errors='coerce')
    df['SCHEDULED_ARRIVAL_HOUR'] = df['Scheduled_ARR_EST'].dt.hour
    df['SCHEDULED_ARRIVAL_MINUTE'] = df['Scheduled_ARR_EST'].dt.minute

    # Isolate the time that the airplane is scheduled to arrived at origin airport
    df['Scheduled_ARR_Ori'] = pd.to_datetime(df['Scheduled_ARR_Ori'], errors='coerce')
    df['SCHEDULED_ORIGIN_ARRIVAL_HOUR'] = df['Scheduled_ARR_Ori'].dt.hour
    df['SCHEDULED_ORIGIN_ARRIVAL_MINUTE'] = df['Scheduled_ARR_Ori'].dt.minute

    # Isolate the time that the airplane actually arrived at origin airport
    df['Actual_ARR_dt_Ori'] = pd.to_datetime(df['Actual_ARR_dt_Ori'], errors='coerce')
    df['ACTUAL_ORIGIN_ARRIVAL_HOUR'] = df['Actual_ARR_dt_Ori'].dt.hour
    df['ACTUAL_ORIGIN_ARRIVAL_MINUTE'] = df['Actual_ARR_dt_Ori'].dt.minute

    # Use one-hot encoding on MKT_CARRIER, OP-CARRIER, ORIGIN, DEST, FAA_CLASS, day_of_week
    df = pd.get_dummies(df, columns=['OP_CARRIER', 'ORIGIN', 'DEST', 'FAA_class', 'day_of_week', 'MKT_CARRIER'])

    # Use on hot encoding with drop first
    df = pd.get_dummies(df, columns=['late_airjet_when_turnaround_within_180'], drop_first=True)

    # Delta Arrival time (interaction term)
    df['DT_ARRIVAL_HOUR'] = df['SCHEDULED_ORIGIN_ARRIVAL_HOUR'] - df['ACTUAL_ORIGIN_ARRIVAL_HOUR']
    df['DT_ARRIVAL_MINUTE'] = df['SCHEDULED_ORIGIN_ARRIVAL_MINUTE'] - df['ACTUAL_ORIGIN_ARRIVAL_MINUTE']

    # Time on tarmac is squared if the high risk flag is true
    df['scheduled_Turnarnd'] = np.where(
        df['late_airjet_when_turnaround_within_180_1'],  # condition column
        df['scheduled_Turnarnd'] ** 2,                 # value if True
        df['scheduled_Turnarnd']                       # value if False
    )

    # Remove unnecessary columns
    df.drop(columns=['Scheduled_DEP', 'Scheduled_ARR_Local', 'CRS_DEP_1hrpre', 'CRS_DEP_1hrpost', 'Scheduled_ARR_Ori', 'Actual_ARR_dt_Ori', 'Scheduled_DEP_EST', 'Scheduled_ARR_EST', 'Scheduled_ARR_Local'], axis=1, inplace=True)

    return df

# Clean the data
df_train_cleaned = make_cleaned_data(df_train)
df_test_cleaned = make_cleaned_data(df_test)

# Create the datasets
X_train = df_train_cleaned.drop('DEP_DEL15', axis=1)
X_test = df_test_cleaned.drop('ID', axis=1)
y_train = df_train_cleaned['DEP_DEL15']

# Align columns between training and test sets
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

X_train = X_train.fillna(0)
X_test = X_test.fillna(0)

# Train the model
model = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)

model.fit(X_train, y_train)

# Create the predictions
y_pred = model.predict(X_test)

submission = pd.DataFrame({'ID': pd.Series(df_test_cleaned['ID']), 'DEP_DEL15':pd.Series(y_pred)})
submission.to_csv('submission.csv', index=False)
for _, row in submission.iterrows():
    print(f"ID: {row['ID']}, DEP_DEL15: {row['DEP_DEL15']}")

print("Submission file successfully created!")

/usr/lib/python3.13/site-packages/xgboost/training.py:183: UserWarning: [12:23:07] WARNING: /usr/src/debug/python-xgboost/xgboost-3.0.4/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


ID: 152931, DEP_DEL15: 0
ID: 4566748, DEP_DEL15: 1
ID: 3264510, DEP_DEL15: 0
ID: 1885080, DEP_DEL15: 1
ID: 5501259, DEP_DEL15: 0
ID: 2039156, DEP_DEL15: 0
ID: 1120359, DEP_DEL15: 0
ID: 1635150, DEP_DEL15: 0
ID: 4766512, DEP_DEL15: 1
ID: 3972901, DEP_DEL15: 1
ID: 4376791, DEP_DEL15: 0
ID: 472751, DEP_DEL15: 0
ID: 1472021, DEP_DEL15: 0
ID: 5178383, DEP_DEL15: 0
ID: 1815184, DEP_DEL15: 0
ID: 544573, DEP_DEL15: 0
ID: 1677722, DEP_DEL15: 0
ID: 4627038, DEP_DEL15: 0
ID: 4543593, DEP_DEL15: 0
ID: 4825733, DEP_DEL15: 1
ID: 3074806, DEP_DEL15: 0
ID: 288914, DEP_DEL15: 0
ID: 915059, DEP_DEL15: 1
ID: 4002630, DEP_DEL15: 0
ID: 3303419, DEP_DEL15: 0
ID: 5187790, DEP_DEL15: 0
ID: 348850, DEP_DEL15: 0
ID: 434630, DEP_DEL15: 0
ID: 2585457, DEP_DEL15: 0
ID: 476323, DEP_DEL15: 0
ID: 3630105, DEP_DEL15: 0
ID: 3807434, DEP_DEL15: 0
ID: 2983556, DEP_DEL15: 1
ID: 884152, DEP_DEL15: 0
ID: 2086110, DEP_DEL15: 1
ID: 863721, DEP_DEL15: 0
ID: 3358693, DEP_DEL15: 0
ID: 5064262, DEP_DEL15: 0
ID: 2171657, DEP_DEL15